![Databricks Academy](./Includes/images/common/db-academy.png)

# 4 Lab - Create a Pipeline  
### Estimated Duration: ~15-20 minutes

In this lab, you'll migrate a traditional ETL workflow to a pipeline for incremental data processing. You'll practice building streaming tables and materialized views using Lakeflow Spark Declarative Pipelines syntax.

#### Your Tasks:
- Create a new pipeline  
- Convert traditional SQL ETL to declarative syntax for incremental processing 
- Configure pipeline settings  
- Define data quality expectations  
- Validate and run the pipeline

### Learning Objectives

By the end of this lab, you will be able to:
- Create a pipeline and execute it successfully using the Lakeflow Pipeline Editor.
- Modify and configure pipeline settings to align with specific data processing requirements.
- Integrate data quality expectations into a pipeline and evaluate their effectiveness.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**  
![Serverless Select](./Includes/images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:** This notebook was **developed and tested using Serverless V5**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>

## A. Classroom Setup

Run the following cell to configure your working environment for this lab.

In [0]:
%run ./Includes/Classroom-Setup-Lab-basic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.8/832.8 kB 10.5 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-16276566-47a2-4c19-857a-83b4607b732e
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.67.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-16276566-47a2-4c19-857a-83b4607b732e
    Can't uninstall 'databricks-sdk'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.65.0 requires protobuf!=3.20.0,

✅ Vocareum workspace detected.
✅ Using existing Vocareum catalog: 'labuser15140516_1778971530'.



  STEP 1: Verifying catalog exists: labuser15140516_1778971530
  Catalog 'labuser15140516_1778971530' exists.

  STEP 2: Setting up 3 schema(s) in catalog: labuser15140516_1778971530
  [1/3] Checking: `labuser15140516_1778971530`.`sdp_lab_1_bronze`... CREATED
  [2/3] Checking: `labuser15140516_1778971530`.`sdp_lab_2_silver`... CREATED
  [3/3] Checking: `labuser15140516_1778971530`.`sdp_lab_3_gold`... CREATED

  COMPLETE: 3 schema(s) created, 0 already existed.



Creating volumes in: labuser15140516_1778971530.sdp_lab_1_bronze

  ✓ labuser15140516_1778971530.sdp_lab_1_bronze.lab_files (created)

Done. 1 volume(s) processed.



  Searching for 'Includes/data' folder...
  Current directory: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines
  Checking: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines/Includes/data... FOUND


  STEP 1: Validating volume folder path...
  Found: /Volumes/labuser15140516_1778971530/sdp_lab_1_bronze/lab_files/

  STEP 2: Scanning for files...
  No files found in: /Volumes/labuser15140516_1778971530/sdp_lab_1_bronze/lab_files/


  STEP 1: Validating source workspace folder...
  Source folder found: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines/Include

Information,Value
Your Catalog:,labuser15140516_1778971530
Bronze Schema:,sdp_lab_1_bronze
Silver Schema:,sdp_lab_2_silver
Gold Schema:,sdp_lab_3_gold
Source Volume:,/Volumes/labuser15140516_1778971530/sdp_lab_1_bronze/lab_files/


Compute,Status,Details
Serverless,✓ Match,Version 5


## B. Cenário

Sua equipe de engenharia de dados identificou uma oportunidade para modernizar um pipeline ETL existente que foi originalmente desenvolvido em um notebook do Databricks. Embora o pipeline atual cumpra seu papel, ele não possui a escalabilidade, observabilidade, eficiência e recursos automatizados de qualidade de dados necessários à medida que o volume e a complexidade dos dados aumentam.

Para resolver isso, você foi solicitado a migrar o pipeline existente para um Lakeflow Spark Declarative Pipeline. Os Spark Declarative Pipelines permitirão que sua equipe defina as transformações de dados de forma mais declarativa, aplique regras de qualidade de dados e aproveite otimização integrada, rastreamento de linhagem e monitoramento.

Seu objetivo é refatorar a lógica original baseada em notebook (mostrada nas células abaixo) para um Spark Declarative Pipeline.

### Requisitos:
  - Migrar o código ETL abaixo para um Spark Declarative Pipeline.
  - Adicionar as expectativas de qualidade de dados necessárias à tabela bronze e à tabela silver.
  - Criar views materializadas para as informações agregadas mais atualizadas.

Siga os passos abaixo para completar sua tarefa.

<div style="max-width: 1200px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #F9F7F4; border-radius: 10px; padding: 22px 26px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid #FF5F46;">

  <img src="./Includes/images/common/genie-code.png" style="height: 44px; margin-bottom: 10px;">

  <div style="font-size: 18pt; font-weight: 700; color: #0b2026; margin-bottom: 12px;">
    Need Help? Use Genie Code
  </div>

  <div style="font-size: 15pt; color: #0b2026; line-height: 1.6; margin-bottom: 16px;">
    Genie is an AI-powered assistant that can help you as you work through this lab. 
    Use it if you get stuck or want a little extra guidance.
  </div>

  <a href="https://docs.databricks.com/aws/en/genie-code/" target="_blank" style="display: inline-block; background: #1B5162; color: white; font-size: 14pt; font-weight: 700; padding: 10px 22px; border-radius: 8px; text-decoration: none;">AWS</a> 
  <a href="https://learn.microsoft.com/en-us/azure/databricks/genie-code/" target="_blank" style="display: inline-block; background: #1B5162; color: white; font-size: 14pt; font-weight: 700; padding: 10px 22px; border-radius: 8px; text-decoration: none;">Azure</a> 
  <a href="https://docs.databricks.com/gcp/en/genie-code/" target="_blank" style="display: inline-block; background: #1B5162; color: white; font-size: 14pt; font-weight: 700; padding: 10px 22px; border-radius: 8px; text-decoration: none;">GCP</a>

</div>

</div>

### B1. Explore the Raw Data

1. Complete the following steps to view where the lab's streaming raw source files are coming from:

   a. Select the **Catalog** icon ![Catalog Icon](./Includes/images/common/catalog_icon.png) in the left navigation bar.  
   
   b. Expand your **labuser** catalog.  
   
   c. Expand the **sdp_lab_1_bronze** schema.  
   
   d. Expand the **lab_files** volume.  
   
   e. You should see a single CSV file named **employees_1.csv**. If not, refresh the catalog.  
   
   f. The files in the **lab_files** volume will be the data source files you will be ingesting.

2. Run the cell below to view the raw CSV file in your **lab_files** volume. Notice the following:

   - It is a simple CSV file separated by commas.  
   - It contains headers.  
   - It has 7 rows in total (6 data records and 1 header row).  
   - The first record (row 2) is a test record and should not be included in the pipeline. It will be dropped by a data quality expectation later.

In [0]:
spark.sql(f'''
        SELECT *
        FROM csv.`/Volumes/{my_catalog}/sdp_lab_1_bronze/lab_files/`
        ''').display()


_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7
EmployeeID,FirstName,Country,Department,Salary,HireDate,Operation,ProcessDate
null,test,test,test,9999,2025-01-01,new,2025-06-05
1,Sophia,US,Sales,72000,2025-04-01,new,2025-06-05
2,Nikos,Gr,IT,55000,2025-04-10,new,2025-06-05
3,Liam,US,Sales,69000,2025-05-03,new,2025-06-05
4,Elena,GR,IT,53000,2025-06-04,new,2025-06-05
5,James,Us,IT,60000,2025-06-05,new,2025-06-05


### B2. Current ETL Code

Run each cell below to view the results of the current ETL pipeline. This will give you an idea of the expected output. Don't worry too much about the data transformations within the SQL queries.

The focus of this lab is on using **declarative SQL** and creating a **Spark Declarative Pipeline**. You will not need to modify the transformation logic. 

You will only need to modify the `CREATE` statements and `FROM` clauses to ensure data is read and processed incrementally in your pipeline.

#### B2.1 - CSV para Bronze

Explore o código e execute a célula. Observe os resultados. Note que:

- O arquivo CSV no volume é lido como uma tabela chamada **employees_bronze_lab4** no esquema **labuser.sdp_lab_1_bronze**.  
- A tabela contém 6 linhas com os nomes de coluna corretos.

Pense sobre o que você precisará mudar ao migrar isso para um Spark Declarative Pipeline. Dicas são adicionadas como comentários no código abaixo.

**NOTA:** No seu Spark Declarative Pipeline, você vai querer adicionar expectativas de qualidade de dados para documentar qualquer dado ruim entrando no pipeline.

In [0]:
%sql
-- Specify to use your labuser catalog
USE CATALOG IDENTIFIER(my_catalog);

In [0]:
%sql
CREATE OR REPLACE TABLE sdp_lab_1_bronze.employees_bronze_lab4
AS
SELECT 
  *,
  current_timestamp() AS ingestion_time,
  _metadata.file_name AS raw_file_name
FROM read_files(
  '/Volumes/' || my_catalog || '/sdp_lab_1_bronze/lab_files',
  format => 'CSV',
  -- Explicit schema
  schema => '
    EmployeeID STRING,
    FirstName STRING,
    Country STRING,
    Department STRING,
    Salary DOUBLE,
    HireDate DATE,
    Operation STRING,
    ProcessDate DATE
  ',
  -- CSV parsing options
  header => 'true',
  inferSchema => 'false'
);

-- Display table
SELECT *
FROM sdp_lab_1_bronze.employees_bronze_lab4;

EmployeeID,FirstName,Country,Department,Salary,HireDate,Operation,ProcessDate,ingestion_time,raw_file_name
null,test,test,test,9999.0,2025-01-01,new,2025-06-05,2026-05-17T15:32:42.350Z,employees_1.csv
1,Sophia,US,Sales,72000.0,2025-04-01,new,2025-06-05,2026-05-17T15:32:42.350Z,employees_1.csv
2,Nikos,Gr,IT,55000.0,2025-04-10,new,2025-06-05,2026-05-17T15:32:42.350Z,employees_1.csv
3,Liam,US,Sales,69000.0,2025-05-03,new,2025-06-05,2026-05-17T15:32:42.350Z,employees_1.csv
4,Elena,GR,IT,53000.0,2025-06-04,new,2025-06-05,2026-05-17T15:32:42.350Z,employees_1.csv
5,James,Us,IT,60000.0,2025-06-05,new,2025-06-05,2026-05-17T15:32:42.350Z,employees_1.csv


#### B2.2 - Bronze to Silver

1. Run the cell below to create the table **labuser.sdp_lab_1_bronze.employees_silver_lab4** and explore the results. 

    Notice that a few simple data transformations were applied to the bronze table and metadata columns were removed.

    Think about what you will need to change when migrating this to a Spark Declarative Pipeline. Hints are added as comments in the code below.

    **NOTE:** For simplicity, we are leaving the **test** row in place, and you will remove it using data quality expectations. Typically, we could have just filtered out the null value(s).

In [0]:
%sql
CREATE OR REPLACE TABLE sdp_lab_2_silver.employees_silver_lab4 -- You will have to modify this to create a streaming table in the pipeline
AS
SELECT
  EmployeeID,
  FirstName,
  upper(Country) AS Country,
  Department,
  Salary,
  HireDate,
  date_format(HireDate, 'MMMM') AS HireMonthName,
  year(HireDate) AS HireYear, 
  Operation
FROM sdp_lab_1_bronze.employees_bronze_lab4;  -- You will have to modify FROM clause to incrementally read in data


-- Display table
SELECT *
FROM sdp_lab_2_silver.employees_silver_lab4;

EmployeeID,FirstName,Country,Department,Salary,HireDate,HireMonthName,HireYear,Operation
null,test,TEST,test,9999.0,2025-01-01,January,2025,new
1,Sophia,US,Sales,72000.0,2025-04-01,April,2025,new
2,Nikos,GR,IT,55000.0,2025-04-10,April,2025,new
3,Liam,US,Sales,69000.0,2025-05-03,May,2025,new
4,Elena,GR,IT,53000.0,2025-06-04,June,2025,new
5,James,US,IT,60000.0,2025-06-05,June,2025,new


#### B2.3 - Silver to Gold
The code below creates two traditional views to aggregate the silver table.

1. Run the cell to create a view that calculates the **total number of employees and total salary by country**.

    Think about what you will need to change when migrating this to a Spark Declarative Pipeline. A hint is added as a comment in the code below.

In [0]:
%sql
CREATE OR REPLACE VIEW sdp_lab_3_gold.employees_by_country_gold_lab4 -- You will have to modify this to create a materialized view in the pipeline
AS
SELECT 
  Country,
  count(*) AS TotalEmployees,
  sum(Salary) AS TotalSalary
FROM sdp_lab_2_silver.employees_silver_lab4 -- You will have to modify FROM clause to incrementally read in data
GROUP BY Country;


-- Display view
SELECT *
FROM sdp_lab_3_gold.employees_by_country_gold_lab4;

Country,TotalEmployees,TotalSalary
TEST,1,9999.0
US,3,201000.0
GR,2,108000.0


2. Run the cell to create a view that calculates the **total salary by department**.

    Think about what you will need to change when migrating this to a Spark Declarative Pipeline. A hint is added as a comment in the code below.

In [0]:
%sql
CREATE OR REPLACE VIEW sdp_lab_3_gold.salary_by_department_gold_lab4  -- You will have to modify this to create a materialized view in the pipeline
AS
SELECT
  Department,
  sum(Salary) AS TotalSalary
FROM sdp_lab_2_silver.employees_silver_lab4
GROUP BY Department;


-- Display view
SELECT *
FROM sdp_lab_3_gold.salary_by_department_gold_lab4;

Department,TotalSalary
test,9999.0
Sales,141000.0
IT,168000.0


#### B2.4 - Delete the Tables

Run the cell below to delete all the tables you created above. You will recreate them as streaming tables and materialized views in the Spark Declarative Pipeline.

In [0]:
%sql
DROP TABLE IF EXISTS sdp_lab_1_bronze.employees_bronze_lab4;
DROP TABLE IF EXISTS sdp_lab_2_silver.employees_silver_lab4;
DROP VIEW IF EXISTS sdp_lab_3_gold.employees_by_country_gold_lab4;
DROP VIEW IF EXISTS sdp_lab_3_gold.salary_by_department_gold_lab4;

Run the cell below to view and copy the path to your **lab_files** volume. You will need this path when building your pipeline to reference your data source files.

**NOTE:** You can also navigate to the volume and copy the path using the UI.

In [0]:
print(f'/Volumes/{my_catalog}/sdp_lab_1_bronze/lab_files')

/Volumes/labuser15140516_1778971530/sdp_lab_1_bronze/lab_files


## C. TO DO: Create the Lakeflow Spark Declarative Pipeline (Steps)

Now that you have explored the traditional ETL code used to create the tables and views, it's time to modify that syntax to declarative SQL for your new pipeline.

You will need to complete the following:

### C1. Criar o Spark Declarative Pipeline

1. Para criar o pipeline e adicionar ativos existentes para associá-lo a arquivos de código já disponíveis em seu Workspace (incluindo pastas Git), complete o seguinte:

   a. Para facilitar, abra **Jobs & Pipelines** em uma nova aba:

    - Na barra de navegação principal, clique com o botão direito em **Jobs & Pipelines** e selecione **Abrir em uma nova aba**.

   b. Em **Jobs & Pipelines** selecione **Create** → **ETL Pipeline**.

   c. Selecione **Settings (ou o ícone de engrenagem)** e complete o seguinte:

 Seção | Campo | Valor |
---------|-------|---------------|
 **Configurações do pipeline** |  **Nome** | `Lab4 - projeto de pipeline firstname` |
 **Local padrão para ativos de dados** | **Catálogo padrão** | Seu catálogo **labuser** |
 **Local padrão para ativos de dados** | **Schema padrão** | Seu schema (database) **sdp_lab_1_bronze** |


**NOTA:** No pipeline, navegue até a pasta de transformações e localize o arquivo my_transformation.py. Abra o menu de três pontos ao lado do arquivo, selecione Renomear e altere o nome do arquivo de `my_transformation.py` para `my_transformation.sql`.

### C2. Criar a Tabela Bronze

1. Migre o código ETL (mostrado abaixo para cada etapa como referência) para um ou mais arquivos e pastas para organizar seu pipeline (você também pode colocar tudo em um único arquivo se preferir).

<br></br>
Modifique o código mostrado abaixo para criar a **tabela bronze** de streaming em seu pipeline, completando o seguinte:

- Modifique a instrução `CREATE OR REPLACE TABLE` para criar uma tabela de streaming.

- Adicione a palavra-chave `STREAM` na cláusula `FROM` para ingerir dados incrementalmente do volume.

- Atualize o caminho na função `read_files()` para apontar para o caminho do volume **labuser.sdp_lab_1_bronze.lab_files** (exemplo: `/Volumes/labuser1234/sdp_lab_1_bronze/lab_files`).
    - Você pode adicionar o caminho estaticamente na função `read_files` ou usar um parâmetro de configuração.

<br></br>
SQL
CREATE OR REPLACE TABLE sdp_lab_1_bronze.employees_bronze_lab4
AS
SELECT 
  *,
  current_timestamp() AS ingestion_time,
  _metadata.file_name AS raw_file_name
FROM read_files(
  'YOUR_PATH_TO_YOUR_lab_files_VOLUME', --Adicione o caminho para seu volume lab_files
  format => 'CSV',
  -- Esquema explícito
  schema => '
    EmployeeID STRING,
    FirstName STRING,
    Country STRING,
    Department STRING,
    Salary DOUBLE,
    HireDate DATE,
    Operation STRING,
    ProcessDate DATE
  ',
  -- Opções de parsing CSV
  header => 'true',
  inferSchema => 'false'
);


2. Faça um **Dry Run** para confirmar se a sintaxe está correta.

##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
CREATE OR REFRESH STREAMING TABLE sdp_lab_1_bronze.employees_bronze_lab4
AS
SELECT 
  *,
  current_timestamp() AS ingestion_time,
  _metadata.file_name AS raw_file_name
FROM STREAM read_files(
  '/Volumes/ADD_YOUR_CATALOG_NAME/sdp_lab_1_bronze/lab_files', --Add the path to your lab_files volume
  format => 'CSV',
  -- Explicit schema
  schema => '
    EmployeeID STRING,
    FirstName STRING,
    Country STRING,
    Department STRING,
    Salary DOUBLE,
    HireDate DATE,
    Operation STRING,
    ProcessDate DATE
  ',
  -- CSV parsing options
  header => 'true',
  inferSchema => 'false'
);

<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>

### C3. Create the Silver Table

1. Modify the code shown below to create the **silver** streaming table by completing the following in your pipeline project:

- Modify the `CREATE OR REPLACE TABLE` statement to create a streaming table.  

- Add the keyword `STREAM` in the `FROM` clause to incrementally ingest data.

- Add the following data quality expectations:      
    ```
    CONSTRAINT check_country EXPECT (Country IN ('US','GR')),
    CONSTRAINT check_salary EXPECT (Salary > 0),
    CONSTRAINT check_null_id EXPECT (EmployeeID IS NOT NULL) ON VIOLATION DROP ROW

    ```


```
CREATE OR REPLACE TABLE sdp_lab_2_silver.employees_silver_lab4 -- You will have to modify this to create a streaming table in the pipeline
AS
SELECT
  EmployeeID,
  FirstName,
  upper(Country) AS Country,
  Department,
  Salary,
  HireDate,
  date_format(HireDate, 'MMMM') AS HireMonthName,
  year(HireDate) AS HireYear, 
  Operation
FROM sdp_lab_1_bronze.employees_bronze_lab4;  -- You will have to modify FROM clause to incrementally read in data
```


2. Try a **Dry Run** to confirm the syntax is correct.

##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
CREATE OR REFRESH STREAMING TABLE sdp_lab_2_silver.employees_silver_lab4 -- You will have to modify this to create a streaming table in the pipeline
(
    CONSTRAINT check_country EXPECT (Country IN ('US','GR')),
    CONSTRAINT check_salary EXPECT (Salary > 0),
    CONSTRAINT check_null_id EXPECT (EmployeeID IS NOT NULL) ON VIOLATION DROP ROW
)
AS
SELECT
  EmployeeID,
  FirstName,
  upper(Country) AS Country,
  Department,
  Salary,
  HireDate,
  date_format(HireDate, 'MMMM') AS HireMonthName,
  year(HireDate) AS HireYear, 
  Operation
FROM STREAM sdp_lab_1_bronze.employees_bronze_lab4;  -- You will have to modify FROM clause to incrementally read in data
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>


### C4. Create the Gold Materialized Views

1. Replace the `CREATE OR REPLACE VIEW` statement in both views to **create materialized views** instead of traditional views in your Spark Declarative Pipeline.

```
-- Employee by Country
CREATE OR REPLACE VIEW sdp_lab_3_gold.employees_by_country_gold_lab4 -- You will have to modify this to create a materialized view in the pipeline
AS
SELECT 
  Country,
  count(*) AS TotalEmployees,
  sum(Salary) AS TotalSalary
FROM sdp_lab_2_silver.employees_silver_lab4
GROUP BY Country;

-- Salary by Department
CREATE OR REPLACE VIEW sdp_lab_3_gold.salary_by_department_gold_lab4  -- You will have to modify this to create a materialized view in the pipeline
AS
SELECT
  Department,
  sum(Salary) AS TotalSalary
FROM sdp_lab_2_silver.employees_silver_lab4
GROUP BY Department;
```

2. Try a **Dry Run** to confirm the syntax is correct.

##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
-- Employee by Country
CREATE OR REFRESH MATERIALIZED VIEW sdp_lab_3_gold.employees_by_country_gold_lab4 -- You will have to modify this to create a materialized view in the pipeline
AS
SELECT 
  Country,
  count(*) AS TotalEmployees,
  sum(Salary) AS TotalSalary
FROM sdp_lab_2_silver.employees_silver_lab4
GROUP BY Country;

-- Salary by Department
CREATE OR REFRESH MATERIALIZED VIEW sdp_lab_3_gold.salary_by_department_gold_lab4  -- You will have to modify this to create a materialized view in the pipeline
AS
SELECT
  Department,
  sum(Salary) AS TotalSalary
FROM sdp_lab_2_silver.employees_silver_lab4
GROUP BY Department;
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>

### C5. Modify the Pipeline Settings

1. Pipeline configuration requirements:

- Your Spark Declarative Pipeline should use **Serverless** compute.  

- Your pipeline should use your **labuser** catalog by default.  

- Your pipeline should use your **sdp_lab_1_bronze** schema by default.

- Make sure your pipeline is including your files.

- **(OPTIONAL)** If using a configuration variable to reference your volume path, make sure it is defined and applied in the `read_files()` function.

### C6. Run the Pipeline
1. When complete, run the pipeline. Troubleshoot any errors.

<br></br>

##### Final Spark Declarative Pipeline Image  
Below is what your final pipeline should look like after the first run with a single CSV file.

![Final Lab4 Pipeline](./Includes/images/lab-1/lab-1-run-2-one-file.png)

## D. Explore the Streaming Tables and Materialized Views

After you have created and run your Spark Declarative Pipeline, complete the following tasks to explore your new streaming tables and materialized views.

1. In the Catalog Explorer on the left, expand your **labuser** catalog and expand the following schemas:

   - **sdp_lab_1_bronze** schema
      - **employees_bronze_lab4** streaming table

   - **sdp_lab_2_silver** schema
      - **employees_silver_lab4** streaming table
      - **country_lookup** UC table (not used in this lab)

   - **sdp_lab_3_gold** schema
      - **employees_by_country_gold_lab4** materialized view
      - **salary_by_department_gold_lab4** materialized view

2. Run the cell below to view the data in your **labuser.sdp_lab_1_bronze.employees_bronze_lab4** streaming table. 

    Notice that the:
    - first row contains a `null` **EmployeeID**.
    - table contains a total of 6 rows.

In [0]:
%sql
SELECT *
FROM sdp_lab_1_bronze.employees_bronze_lab4;

EmployeeID,FirstName,Country,Department,Salary,HireDate,Operation,ProcessDate,ingestion_time,raw_file_name
null,test,test,test,9999.0,2025-01-01,new,2025-06-05,2026-05-17T15:50:19.072Z,employees_1.csv
1,Sophia,US,Sales,72000.0,2025-04-01,new,2025-06-05,2026-05-17T15:50:19.072Z,employees_1.csv
2,Nikos,Gr,IT,55000.0,2025-04-10,new,2025-06-05,2026-05-17T15:50:19.072Z,employees_1.csv
3,Liam,US,Sales,69000.0,2025-05-03,new,2025-06-05,2026-05-17T15:50:19.072Z,employees_1.csv
4,Elena,GR,IT,53000.0,2025-06-04,new,2025-06-05,2026-05-17T15:50:19.072Z,employees_1.csv
5,James,Us,IT,60000.0,2025-06-05,new,2025-06-05,2026-05-17T15:50:19.072Z,employees_1.csv


3. Run the cell below to view the data in your **labuser.sdp_lab_2_silver.employees_silver_lab4** streaming table. 

    Notice that the silver table:
    - removed the row where **EmployeeID** was `null` using a data quality expectation.
    - contains a total of 5 rows.

In [0]:
%sql
SELECT *
FROM sdp_lab_2_silver.employees_silver_lab4;

EmployeeID,FirstName,Country,Department,Salary,HireDate,HireMonthName,HireYear,Operation
1,Sophia,US,Sales,72000.0,2025-04-01,April,2025,new
2,Nikos,GR,IT,55000.0,2025-04-10,April,2025,new
3,Liam,US,Sales,69000.0,2025-05-03,May,2025,new
4,Elena,GR,IT,53000.0,2025-06-04,June,2025,new
5,James,US,IT,60000.0,2025-06-05,June,2025,new


4. Run the cell below to view the data in your **labuser.sdp_lab_3_gold.employees_by_country_gold_lab4** materialized view. 

    **Final Results**
    | Country | TotalCount | TotalSalary |
    |---------|------------|-------------|
    | GR      | 2          | 108000      |
    | US      | 3          | 201000      |

In [0]:
%sql
SELECT *
FROM sdp_lab_3_gold.employees_by_country_gold_lab4
ORDER BY TotalSalary;

Country,TotalEmployees,TotalSalary
GR,2,108000.0
US,3,201000.0


5. Run the cell below to view the data in your **labuser.sdp_lab_3_gold.salary_by_department_gold_lab4** materialized view. 

    **Final Results**
    | Department  | TotalSalary |
    |-------------|-------------|
    | Sales       | 141000      |
    | IT          | 168000      |

In [0]:
%sql
SELECT *
FROM sdp_lab_3_gold.salary_by_department_gold_lab4
ORDER BY TotalSalary;

Department,TotalSalary
Sales,141000.0
IT,168000.0


## E. Challenge Scenario (Optional in Live Class)
### Duration: ~10 minutes

**NOTE:** *If you finish early in a live class, feel free to complete the challenge below. The challenge is optional and most likely will not be completed during the live class. Only continue if your Spark Declarative Pipeline was set up correctly in the previous section by comparing your pipeline to the solution image.*

**SCENARIO:** In this challenge, you will land a new CSV file in your **lab_files** cloud storage volume and rerun the pipeline to observe that the Spark Declarative Pipeline only ingests the new data.

### E1. Land Another CSV File in Cloud Storage and Preview

1. Run the cell below to copy another file to your **labuser.sdp_lab_1_bronze.lab_files** volume.


In [0]:
## Find data in workspace data folder
data_path = find_folder('Includes/data')

## Land another CSV file to your lab_files volume
copy_workspace_files_to_volume(
    src_workspace_folder=f'{data_path}/lab_files',
    target_volume_path=f'/Volumes/{my_catalog}/sdp_lab_1_bronze/lab_files',
    n=2
)


  Searching for 'Includes/data' folder...
  Current directory: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines
  Checking: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines/Includes/data... FOUND


  STEP 1: Validating source workspace folder...
  Source folder found: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines/Includes/data/lab_files

  STEP 2: Checking target volume path...
  Target volume path already exists: /Volumes/labuser15140516_1778971530/sdp_lab_1_bronze/lab_files

  STEP 3: Reading source files...
  Found 3 file(s) in source folder.

  STE

2. In the left navigation area, navigate to your **labuser.sdp_lab_1_bronze.lab_files** volume and expand it. 

    Confirm it contains two CSV files: 
    - **employees_1.csv** 
    - **employees_2.csv**

**NOTE:** You may need to refresh your catalog if the file is not shown.

3. Run the cell below to preview only the new CSV file (`employees_2.csv`) and view the results. Notice that the new CSV file contains employee information:

    - Contains 4 rows.  
    - The **Operation** column specifies an action for each employee (e.g., update the record, delete the record, or add a new employee).

**Output**
| EmployeeID | FirstName | Country | Department | Salary | HireDate   | Operation | ProcessDate |
|------------|----------|---------|------------|--------|------------|-----------|-------------|
| 6          | Emily    | us      | Enablement | 80000  | 2025-06-09 | new       | 2025-06-22  |
| 7          | Yannis   | gR      | HR         | 70000  | 2025-06-20 | new       | 2025-06-22  |
| 3          | Liam     | US      | Sales      | 100000 | 2025-05-03 | update    | 2025-06-22  |
| 1          | null     | null    | null       | null   | -          | delete    | 2025-06-22  |

**NOTE:** Don't worry about the **Operation** column yet. We'll cover how to capture these specific changes in your data (Change Data Capture) in a later demonstration.

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/' || my_catalog || '/sdp_lab_1_bronze/lab_files/employees_2.csv',
  format => 'CSV',
  schema => '
    EmployeeID STRING,
    FirstName STRING,
    Country STRING,
    Department STRING,
    Salary DOUBLE,
    HireDate DATE,
    Operation STRING,
    ProcessDate DATE
  ',
  -- CSV parsing options
  header => 'true',
  inferSchema => 'false'
)

EmployeeID,FirstName,Country,Department,Salary,HireDate,Operation,ProcessDate
6,Emily,us,Enablement,80000.0,2025-06-09,new,2025-06-22
7,Yannis,gR,HR,70000.0,2025-06-20,new,2025-06-22
3,Liam,US,Sales,100000.0,2025-05-03,update,2025-06-22
1,null,null,null,null,null,delete,2025-06-22


### E2. Run the Pipeline with the New CSV File

1. Now that you have explored the new CSV file in cloud storage, go back to your Spark Declarative Pipeline and select **Run pipeline**. 

    Notice that the pipeline incrementally process the new file from cloud storage.


##### Final Spark Declarative Pipeline Image
Below is what your final pipeline should look like after the second run with two CSV files.

![Final Challenge Lab4 DLT Pipeline](./Includes/images/lab-1/lab-1-run-2-challenge.png)

### E3. View the Streaming Tables and Materialized Views

1. View your **bronze** streaming table. It should have a total of **10** rows.

**Output**
| EmployeeID | FirstName | Country | Department | Salary | HireDate   | Operation | ProcessDate | ingestion_time               | raw_file_name   |
|------------|----------|---------|------------|--------|------------|-----------|-------------|------------------------------|-----------------|
| null       | test     | test    | test       | 9999   | 2025-01-01 | new       | 2025-06-05  | 2026-04-08T15:56:55.097+00:00 | employees_1.csv |
| 1          | Sophia   | US      | Sales      | 72000  | 2025-04-01 | new       | 2025-06-05  | 2026-04-08T15:56:55.097+00:00 | employees_1.csv |
| 1          | null     | null    | null       | null   | -          | delete    | 2025-06-22  | 2026-04-08T16:02:47.814+00:00 | employees_2.csv |
| 2          | Nikos    | Gr      | IT         | 55000  | 2025-04-10 | new       | 2025-06-05  | 2026-04-08T15:56:55.097+00:00 | employees_1.csv |
| 3          | Liam     | US      | Sales      | 100000 | 2025-05-03 | update    | 2025-06-22  | 2026-04-08T16:02:47.814+00:00 | employees_2.csv |
| 3          | Liam     | US      | Sales      | 69000  | 2025-05-03 | new       | 2025-06-05  | 2026-04-08T15:56:55.097+00:00 | employees_1.csv |
| 4          | Elena    | GR      | IT         | 53000  | 2025-06-04 | new       | 2025-06-05  | 2026-04-08T15:56:55.097+00:00 | employees_1.csv |
| 5          | James    | Us      | IT         | 60000  | 2025-06-05 | new       | 2025-06-05  | 2026-04-08T15:56:55.097+00:00 | employees_1.csv |
| 6          | Emily    | us      | Enablement | 80000  | 2025-06-09 | new       | 2025-06-22  | 2026-04-08T16:02:47.814+00:00 | employees_2.csv |
| 7          | Yannis   | gR      | HR         | 70000  | 2025-06-20 | new       | 2025-06-22  | 2026-04-08T16:02:47.814+00:00 | employees_2.csv |

In [0]:
%sql
SELECT *
FROM sdp_lab_1_bronze.employees_bronze_lab4
ORDER BY EmployeeID;

2. View your **silver** streaming table. It should have a total of **9 rows**.

**Output**
| EmployeeID | FirstName | Country | Department | Salary | HireDate   | HireMonthName | HireYear | Operation |
|------------|----------|---------|------------|--------|------------|----------------|----------|-----------|
| 1          | Sophia   | US      | Sales      | 72000  | 2025-04-01 | April          | 2025     | new       |
| 1          | null     | null    | null       | null   | -          | null           | null     | delete    |
| 2          | Nikos    | GR      | IT         | 55000  | 2025-04-10 | April          | 2025     | new       |
| 3          | Liam     | US      | Sales      | 100000 | 2025-05-03 | May            | 2025     | update    |
| 3          | Liam     | US      | Sales      | 69000  | 2025-05-03 | May            | 2025     | new       |
| 4          | Elena    | GR      | IT         | 53000  | 2025-06-04 | June           | 2025     | new       |
| 5          | James    | US      | IT         | 60000  | 2025-06-05 | June           | 2025     | new       |
| 6          | Emily    | US      | Enablement | 80000  | 2025-06-09 | June           | 2025     | new       |
| 7          | Yannis   | GR      | HR         | 70000  | 2025-06-20 | June           | 2025     | new       |

In [0]:
%sql
SELECT *
FROM sdp_lab_2_silver.employees_silver_lab4
ORDER BY EmployeeID;

3. Explore the history of your streaming tables using the Catalog Explorer. 

    Notice that there are two **STREAMING UPDATES** to both the **bronze** and **silver** tables.

In [0]:
%sql
DESCRIBE HISTORY sdp_lab_1_bronze.employees_bronze_lab4;

In [0]:
%sql
DESCRIBE HISTORY sdp_lab_2_silver.employees_silver_lab4;


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>